# House Price Prediction

**Tabular Regression Project** — Predict house sale prices.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: King County house sales (Kaggle, ~21K rows)
Target: `price`

## 2 · Project Overview

We predict **house sale prices** using the King County (Seattle, WA) housing dataset. This contains ~21K property transactions with 19 house features including square footage, location, condition, and year built.

## 3 · Learning Objectives

1. Explore a real-world housing dataset.
2. Handle feature selection for regression.
3. Apply gradient-boosting regressors.
4. Use LazyPredict and FLAML.
5. Evaluate with RMSE, MAE, R².

## 4 · Problem Statement

Predict the **sale price** of a house from property features like size, bedrooms, condition, grade, and location.

## 5 · Why This Project Matters

- House price prediction is the canonical regression task.
- Real estate valuation affects buyers, sellers, lenders, and investors.
- Understanding price drivers is immediately actionable.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: harlfoxem/housesalesprediction |
| **Rows** | ~21,613 |
| **Target** | price |
| **Features** | bedrooms, bathrooms, sqft_living, grade, yr_built, lat, long, etc. |

## 7 · Dataset Source and License Notes

- **Source**: King County house sales, publicly available on Kaggle.
- **License**: Public domain.
- **Note**: 2014-2015 transactions.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'price'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [4]:
import kagglehub, glob
try:
    path = kagglehub.dataset_download('harlfoxem/housesalesprediction')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
df.head()

Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\harlfoxem\housesalesprediction\versions\1
Loaded: (21613, 21)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


## 12 · Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min():,.0f}, {df[TARGET].max():,.0f}]')

Missing: 0
Duplicates: 0
Target range: [75,000, 7,700,000]


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df[TARGET].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Price Distribution')
if 'sqft_living' in df.columns:
    axes[0,1].scatter(df['sqft_living'], df[TARGET], alpha=0.1, s=2)
    axes[0,1].set_xlabel('sqft_living'); axes[0,1].set_ylabel('price')
    axes[0,1].set_title('Sqft vs Price')
if 'grade' in df.columns:
    df.groupby('grade')[TARGET].mean().plot.bar(ax=axes[1,0])
    axes[1,0].set_title('Avg Price by Grade')
if 'waterfront' in df.columns:
    df.groupby('waterfront')[TARGET].mean().plot.bar(ax=axes[1,1])
    axes[1,1].set_title('Waterfront Effect')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count    2.161300e+04
mean     5.400881e+05
std      3.671272e+05
min      7.500000e+04
25%      3.219500e+05
50%      4.500000e+05
75%      6.450000e+05
max      7.700000e+06
Name: price, dtype: float64

Skewness: 4.02


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [8]:
# Drop id and date
for col in df.columns:
    if 'id' in col.lower() or 'date' in col.lower():
        df = df.drop(columns=[col])
for col in df.select_dtypes(include='object').columns:
    df = df.drop(columns=[col])
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
print(f'Preprocessed: {df.shape}')

Preprocessed: (21613, 19)


## 17 · Feature Engineering

In [9]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (17290, 18), Test: (4323, 18)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:,.0f}, MAE={baseline_mae:,.0f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=212,540, MAE=127,493, R²=0.7012


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                               Adjusted R-Squared  R-Squared           RMSE  \
Model                                                                         
CatBoostRegressor                        0.898956   0.900777  114822.236797   
XGBRegressor                             0.863322   0.865785  133542.674805   
GradientBoostingRegressor                0.863061   0.865528  133670.366760   
HistGradientBoostingRegressor            0.862558   0.865034  133915.551482   
LGBMRegressor                            0.852777   0.855430  138598.476786   
ExtraTreesRegressor                      0.849262   0.851978  140243.500947   
RandomForestRegressor                    0.837064   0.840000  145807.530231   
BaggingRegressor                         0.824785   0.827942  151201.760499   
PoissonRegressor                         0.736158   0.740912  185542.217191   
KNeighborsRegressor                      0.722502   0.727502  190283.431234   
SGDRegressor                             0.696512   

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=122691.35, MAE=68674.73, R²=0.9004
LightGBM: RMSE=143201.94, MAE=70806.78, R²=0.8644


XGBoost: RMSE=134495.66, MAE=67146.06, R²=0.8803


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  CatBoost              RMSE=122691.35  MAE=68674.73  R²=0.9004
  XGBoost               RMSE=134495.66  MAE=67146.06  R²=0.8803
  LightGBM              RMSE=143201.94  MAE=70806.78  R²=0.8644
  Baseline_LR           RMSE=212539.52  MAE=127493.34  R²=0.7012

Top 3: ['CatBoost', 'XGBoost', 'LightGBM']


## 23 · Final Eval of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
CatBoost: RMSE=122691.35, MAE=68674.73, R²=0.9004
XGBoost: RMSE=134495.66, MAE=67146.06, R²=0.8803
LightGBM: RMSE=143201.94, MAE=70806.78, R²=0.8644


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **sqft_living** and **grade** are the top price drivers.
- **Location** (lat/long) captures neighborhood-level variation.
- **Waterfront** properties command a significant premium.
- These insights help real estate agents and investors identify undervalued properties.

## 26 · Limitations

- Single county, 2-year snapshot.
- No school district, crime, or amenity data.
- Market conditions change over time.
- Luxury homes are harder to predict (sparse data).

## 27 · How to Improve

1. Log-transform price.
2. Add school ratings and crime data.
3. Feature interactions (grade × sqft).
4. Neighborhood clustering features.

## 28 · Production Considerations

- Integrate with MLS for real-time data.
- Regular retraining with new sales.
- Explain predictions for compliance.
- Fair lending law compliance.

## 29 · Common Mistakes

1. Not removing the ID column.
2. Predicting on raw (unprocessed) data.
3. Ignoring location features.
4. Not handling price skewness.

## 30 · Mini Challenge

1. Predict log(price) and compare.
2. Add a `house_age` feature.
3. Build separate models for high vs low price segments.
4. Feature importance comparison across models.

## 31 · Final Summary

- Location and property quality (grade, sqft) dominate price.
- Boosting models outperform linear regression.
- Log-transforming the target is the single biggest improvement opportunity.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
